# 04 MLflow: compare runs, register the champion, reload it by name

Day 2, 11:00 to 12:15 (Lab 2, Part E). Lab sheet: `Day2/labs/lab2-mlflow.md`.

**Before you start:** `machine_guard/train.py` logs to MLflow (Lab 2, steps 2 to 4) and you have run
the 8 training commands in the lab sheet. Keep the MLflow UI open in another browser tab: everything
this notebook does, you can also see there.

Cells marked **TODO** are yours to complete.

In [ ]:
import os
import sys
from pathlib import Path

# Work from the repo root (the folder that holds machine_guard/), wherever Jupyter started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "machine_guard").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working in", ROOT)

In [ ]:
import mlflow
import pandas as pd
from mlflow import MlflowClient

from machine_guard.features import REGISTERED_MODEL_NAME, load_data, split
from machine_guard.train import setup_mlflow

uri = setup_mlflow()          # the same database train.py writes to: mlflow.db in the repo root
client = MlflowClient()
print("Tracking URI:", uri)

## 1. Every run, in one table

`mlflow.search_runs` returns your runs as a pandas table: one row per run, one column per param,
metric and tag. This is yesterday's `experiments_by_hand.md`, filled in by the computer.

In [ ]:
runs = mlflow.search_runs(experiment_names=["machine-guard"])
print(len(runs), "runs")

columns = {
    "tags.mlflow.runName": "run",
    "params.model": "model",
    "params.class_weight": "class_weight",
    "params.engineered": "engineered",
    "metrics.recall": "recall",
    "metrics.precision": "precision",
    "metrics.missed_failures": "missed",
    "metrics.false_alarms": "false_alarms",
    "metrics.cost": "cost",
}
table = runs[["run_id", *columns]].rename(columns=columns)
# TODO: show the table sorted by cost, cheapest first (hint: .sort_values("cost")), rounded to 3 decimals
table

**Compare with yesterday.** Open `results/experiments_by_hand.md` next to this table.
Which three things does this table have that yours didn't? (Think: settings you forgot, the exact code
version, the data version, the saved model.)

## 2. Ask questions of your runs (this is how teams manage hundreds of them)

With 8 runs you can read the table. With 800 you ask questions. `filter_string` uses a small SQL-like language.

In [ ]:
# TODO: find every logistic regression run with fewer than 100 false alarms.
#   Start from this example and change the filter:
#   filter_string="params.model = 'random_forest' and metrics.recall > 0.8"
good_forests = mlflow.search_runs(
    experiment_names=["machine-guard"],
    filter_string="params.model = 'random_forest' and metrics.recall > 0.8",
    order_by=["metrics.cost ASC"],
)
good_forests[["tags.mlflow.runName", "metrics.recall", "metrics.false_alarms", "metrics.cost"]].round(3)

## 3. Choose the champion and register it

The **registry** is the shelf of models that are candidates for production. Registering a run's model
gives it a name (`machine-guard`) and a version number (1, 2, 3...). An **alias** such as `champion` is a
movable label that points at one version. Day 3's API will ask for `models:/machine-guard@champion`,
so whatever the alias points at is what gets served.

Decide first: which run is your champion, and why? Write one sentence in the next cell before you run anything.

*My champion is ... because ...*

In [ ]:
best = table.sort_values("cost").iloc[0]      # TODO: is the cheapest run really the one you want? Change if not.
print("Champion run:", best["run"], best["run_id"])
print(f"recall {best['recall']:.3f}, precision {best['precision']:.3f}, "
      f"missed {best['missed']:.0f}, false alarms {best['false_alarms']:.0f}, cost {best['cost']:.0f}")

In [ ]:
from mlflow.exceptions import MlflowException

# Register once. Running this notebook again later must not move @champion: that is a decision.
try:
    version = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
    print(f"@champion already points at version {version.version}: leaving it where it is")
except MlflowException:  # no champion yet: register the chosen run's model
    # TODO: register the champion's model and point the alias "champion" at it.
    #   Write the two lines here, indented by 4 spaces like the lines below:
    #   version = mlflow.register_model(f"runs:/{best['run_id']}/model", REGISTERED_MODEL_NAME)
    #   client.set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", version.version)


    # Copy the lineage tags from the run onto the model version, so they travel with the model
    run = client.get_run(best["run_id"])
    for key in ("git_commit", "data_md5"):
        client.set_model_version_tag(REGISTERED_MODEL_NAME, version.version, key, run.data.tags[key])

    print(f"Registered {REGISTERED_MODEL_NAME} version {version.version} with alias 'champion'")

Now look in the MLflow UI: click **Models** at the top. You should see `machine-guard`, version 1,
with the alias `@champion`.

## 4. Reload the model by name, not by file

Nobody needs to know where the model file lives. They ask the registry for "the champion".
Restart-proof: this works in a new notebook, a new codespace or, tomorrow, inside the API.

In [ ]:
from sklearn.metrics import confusion_matrix

# TODO: load the champion by its alias. The URI looks like "models:/<name>@<alias>"
champion = ...

X_train, X_test, y_train, y_test = split(load_data())
pred = champion.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"Caught {tp} of {tp + fn} failures, {fp} false alarms")

**Check:** the numbers match the champion's row in the table above. The model is the whole pipeline
(feature engineering, encoding, scaling, forest), so it takes the six raw sensor columns directly:

In [ ]:
print(list(champion.named_steps))
X_test.head(3)

## 5. Lineage: where did this model come from?

An auditor asks: "Which code and which data produced the model you are about to deploy?"

In [ ]:
mv = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
run = client.get_run(mv.run_id)
print("Model version:", mv.version)
print("Run:          ", run.info.run_name, run.info.run_id)
print("Params:       ", run.data.params)
print("Git commit:   ", run.data.tags["git_commit"], "(uncommitted changes)" if run.data.tags.get("git_dirty") == "True" else "")
print("Data md5:     ", run.data.tags["data_md5"])

In the terminal, `git show <first 7 characters of the commit>` shows exactly that version of `train.py`.
The data md5 is the fingerprint of the CSV. After lunch, DVC turns it into a data version you can restore.

**Promotion and rollback, in one line each:**
`client.set_registered_model_alias("machine-guard", "champion", <new version>)` promotes a new model.
Pointing it back at the old version is the rollback. Nothing is copied or deleted.

Save the notebook, then commit it: `git add notebooks/04_mlflow_tracking.ipynb && git commit -m "Compare runs and register the champion"`.